# Perceptrons, Backpropagation and Adversarial Examples

Prepared for Fall 2026. Reference: the notes for Lectures 10 (Perceptrons and Backpropagation) and 11 (Adversarial Examples); page numbers refer to these notes. If you find a bug, please email the TAs or tell us at office hours.

**Before you start.** Everything is numpy and matplotlib. Every figure and number a question asks for must appear in your PDF: to vary a parameter, copy the cell below the question (or loop) rather than re-running it above.

## Utilities

In [ ]:
from __future__ import annotations

import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

%matplotlib inline

# The colours of the lecture notes
RED, BLUE = "#d62728", "#1f77b4"          # labels y = +1 and y = -1
TEAL, GREY = "#1b9e77", "#9e9e9e"         # learnt / final; earlier hyperplanes
OLIVE, PURPLE = "#808000", "#7b3294"      # logistic regression; hidden units
ORANGE = "#e6550d"                        # adversarial perturbation; a wrong label


def two_clouds(separable: bool = True, n_per_class: int = 50, seed: int = 0):
    """Two Gaussian clouds of n_per_class points each; the first cloud has the label y = +1, the second y = -1.

    separable=True:  centres (1, 4) and (3, 2), standard deviation 0.6. Points closer than 0.15 to the midline between the
                     centres, or on its wrong side, are redrawn, so the two classes are linearly separable for every seed.
    separable=False: centres (0.6, 0.4) and (-0.6, -0.4), standard deviation 0.55; the clouds overlap.
    """
    rng = np.random.default_rng(seed)
    if separable:
        centres, sigma, margin = np.array([[1.0, 4.0], [3.0, 2.0]]), 0.6, 0.15
    else:
        centres, sigma, margin = np.array([[0.6, 0.4], [-0.6, -0.4]]), 0.55, None
    X, y = [], []
    for c, label in zip(centres, (1.0, -1.0)):
        points = rng.normal(c, sigma, size=(4 * n_per_class, 2))
        if margin is not None:
            mid = centres.mean(0)
            normal = (centres[0] - centres[1]) / np.linalg.norm(centres[0] - centres[1])
            points = points[label * ((points - mid) @ normal) >= margin]
        X.append(points[:n_per_class])
        y.append(np.full(n_per_class, label))
    return np.concatenate(X), np.concatenate(y)


def disc_and_ring(n_per_class: int = 120, seed: int = 0):
    """n_per_class points uniform in the disc of radius 1 (y = +1) and as many in the ring between radii 1.55 and 2.45 (y = -1)."""
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, 2 * n_per_class)
    r = np.concatenate([np.sqrt(rng.uniform(0, 1, n_per_class)),                       # sqrt: uniform in area, not in radius
                        np.sqrt(rng.uniform(1.55**2, 2.45**2, n_per_class))])
    X = np.stack([r * np.cos(theta), r * np.sin(theta)], 1)
    y = np.concatenate([np.ones(n_per_class), -np.ones(n_per_class)])
    return X, y


def add_radius_feature(X):
    """Append the feature phi = |x|^2 to every point: (x1, x2) -> (x1, x2, x1^2 + x2^2)."""
    return np.column_stack([X, (X**2).sum(1)])


def perceptron(X, y, n_passes: int = 10, seed: int = 0):
    """Rosenblatt's perceptron algorithm, from w = 0, w0 = 0, visiting the examples in a random order in every pass.

    Returns (w, w0), (w_avg, w0_avg), corrections, history:
      (w, w0)          the final weights and threshold;
      (w_avg, w0_avg)  the averaged perceptron, the average of (w, w0) over all the steps (every visit of an example);
      corrections      the number of corrections in each pass (the loop stops after a pass without corrections);
      history          the list of (w, w0) after every correction, starting with (0, 0).
    """
    rng = np.random.default_rng(seed)
    n, d = X.shape
    w, w0 = np.zeros(d), 0.0
    w_sum, w0_sum, n_steps = np.zeros(d), 0.0, 0
    corrections, history = [], [(w.copy(), w0)]
    for _ in range(n_passes):
        k = 0
        for i in rng.permutation(n):
            if y[i] * (w @ X[i] + w0) <= 0:          # misclassified, or on the boundary
                w = w + y[i] * X[i]
                w0 = w0 + y[i]
                k += 1
                history.append((w.copy(), w0))
            w_sum += w
            w0_sum += w0
            n_steps += 1
        corrections.append(k)
        if k == 0:
            break
    return (w, w0), (w_sum / n_steps, w0_sum / n_steps), corrections, history


def linear_accuracy(X, y, w, w0) -> float:
    """Fraction of the examples on their correct side of the hyperplane w.x + w0 = 0 (a point on the plane counts as wrong)."""
    return float(np.mean(y * (X @ w + w0) > 0))


def finite_difference_gradient(f, theta, eps: float = 1e-6):
    """Central differences {f(theta + eps e_j) - f(theta - eps e_j)} / (2 eps) for every component j of the vector theta (float64)."""
    theta = np.array(theta, dtype=float)
    grad = np.zeros_like(theta)
    for j in range(len(theta)):
        step = np.zeros_like(theta)
        step[j] = eps
        grad[j] = (f(theta + step) - f(theta - step)) / (2 * eps)
    return grad


def plot_points(X, y, ax, size: int = 12):
    """Red dots for y = +1 and blue dots for y = -1, with equal axes."""
    ax.scatter(X[y > 0, 0], X[y > 0, 1], c=RED, s=size, label="$y = +1$")
    ax.scatter(X[y < 0, 0], X[y < 0, 1], c=BLUE, s=size, label="$y = -1$")
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_aspect("equal")


def plot_hyperplane(w, w0, ax, colour, lw: float = 1.5, label=None, **kw):
    """Draw the line w.x + w0 = 0 (w has two components) across the current limits of the axes."""
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    if abs(w[1]) >= abs(w[0]) and w[1] != 0:
        xs = np.array([x0, x1])
        ys = -(w[0] * xs + w0) / w[1]
    elif w[0] != 0:
        ys = np.array([y0, y1])
        xs = -(w[1] * ys + w0) / w[0]
    else:
        return                                    # w = 0: no line to draw
    ax.plot(xs, ys, color=colour, lw=lw, label=label, **kw)
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)


def label_hyperplane(w, w0, ax, text, colour):
    """Write `text` near the point where the line w.x + w0 = 0 leaves the axes."""
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    norm = np.linalg.norm(w)
    if norm == 0:
        return
    p = -w0 * np.asarray(w) / norm**2                               # the point of the line closest to the origin
    d = np.array([-w[1], w[0]]) / norm                              # the direction of the line
    s_values = [s for s in [(x0 - p[0]) / d[0] if d[0] else None, (x1 - p[0]) / d[0] if d[0] else None,
                            (y0 - p[1]) / d[1] if d[1] else None, (y1 - p[1]) / d[1] if d[1] else None] if s is not None]
    inside = [s for s in s_values if x0 - 1e-9 <= (p + s * d)[0] <= x1 + 1e-9 and y0 - 1e-9 <= (p + s * d)[1] <= y1 + 1e-9]
    if not inside:
        return
    ends = [p + 0.88 * s * d for s in (min(inside), max(inside))]
    q = max(ends, key=lambda e: (x1 - e[0]) / (x1 - x0) + (y1 - e[1]) / (y1 - y0))   # the end farther from the legend's corner
    ax.text(q[0], q[1], text, color=colour, fontsize=10, fontweight="bold", ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.8))


def plot_level_set(fn, ax, colour, level: float = 0.0, lw: float = 2, label=None, n: int = 300):
    """Draw the curve fn(x) = level over the current limits of the axes (fn takes an array of points, N x 2)."""
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    gx, gy = np.meshgrid(np.linspace(x0, x1, n), np.linspace(y0, y1, n))
    values = fn(np.stack([gx.ravel(), gy.ravel()], 1)).reshape(gx.shape)
    if values.min() < level < values.max():
        ax.contour(gx, gy, values, levels=[level], colors=[colour], linewidths=lw)
    if label is not None:
        ax.plot([], [], color=colour, lw=lw, label=label)

## The perceptron

Rosenblatt's algorithm (pp. 33–37) starts from $\vec w = 0$, $w_0 = 0$, visits the examples $(\vec x_n, y_n \in \{\pm 1\})$ in random order and corrects towards every misclassified one,

$$\text{if } y_n(\vec w\cdot\vec x_n + w_0) \le 0: \qquad \vec w \mapsto \vec w + y_n\vec x_n, \qquad w_0 \mapsto w_0 + y_n,$$

stopping after a pass without corrections. Without a separating hyperplane the weights keep changing; the **averaged perceptron**, the average of $(\vec w, w_0)$ over all the steps, usually separates well. The feature $|\vec x|^2$ turns a hyperplane in $(\vec x, |\vec x|^2)$ into a circle in the plane.

`perceptron(X, y, n_passes, seed)` returns the final `(w, w0)`, the averaged `(w_avg, w0_avg)`, `corrections` per pass and `history`.

In [ ]:
X_sep, y_sep = two_clouds(separable=True)
X_ovl, y_ovl = two_clouds(separable=False)
X_dr, y_dr = disc_and_ring()
Phi_dr = add_radius_feature(X_dr)             # (x1, x2, |x|^2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))

# separable clouds: grey lines are the earlier hyperplanes, teal the final one
visit_seed = 1                                # the random order of the visits; try other values for Question 10.1
(w, w0), (w_avg, w0_avg), corrections, history = perceptron(X_sep, y_sep, n_passes=10, seed=visit_seed)
plot_points(X_sep, y_sep, axes[0])
for j, (w_h, w0_h) in enumerate(history[1:-1]):
    plot_hyperplane(w_h, w0_h, axes[0], GREY, lw=1, label="earlier hyperplanes" if j == 0 else None)
plot_hyperplane(w, w0, axes[0], TEAL, lw=2.5, label="final hyperplane")
axes[0].set_title(f"separable clouds: corrections per pass {corrections}")
print(f"separable clouds:         corrections per pass {corrections}; final hyperplane {linear_accuracy(X_sep, y_sep, w, w0):.0%} correct, "
      f"averaged perceptron {linear_accuracy(X_sep, y_sep, w_avg, w0_avg):.0%}")

# overlapping clouds: grey lines are the last eight hyperplanes, teal the averaged perceptron
(w, w0), (w_avg, w0_avg), corrections, history = perceptron(X_ovl, y_ovl, n_passes=10, seed=visit_seed)
plot_points(X_ovl, y_ovl, axes[1])
for j, (w_h, w0_h) in enumerate(history[-8:]):
    plot_hyperplane(w_h, w0_h, axes[1], GREY, lw=1, label="last eight hyperplanes" if j == 0 else None)
plot_hyperplane(w_avg, w0_avg, axes[1], TEAL, lw=2.5, label="averaged perceptron")
axes[1].set_title("overlapping clouds: it keeps changing")
last_eight = [linear_accuracy(X_ovl, y_ovl, w_h, w0_h) for w_h, w0_h in history[-8:]]
print(f"overlapping clouds:       corrections per pass {corrections}; final hyperplane {linear_accuracy(X_ovl, y_ovl, w, w0):.0%} correct, "
      f"last eight hyperplanes {min(last_eight):.0%} to {max(last_eight):.0%}, averaged perceptron {linear_accuracy(X_ovl, y_ovl, w_avg, w0_avg):.0%}")

# disc and ring with the feature |x|^2: the hyperplane in (x, |x|^2) is a circle in the plane
(w, w0), (w_avg, w0_avg), corrections, history = perceptron(Phi_dr, y_dr, n_passes=10, seed=visit_seed)
plot_points(X_dr, y_dr, axes[2])
plot_level_set(lambda points: add_radius_feature(points) @ w + w0, axes[2], TEAL, label=r"hyperplane in $(\vec x, |\vec x|^2)$")
axes[2].set_title(rf"disc and ring with $\phi = |\vec x|^2$: corrections {corrections}")
print(f"disc and ring with |x|^2: corrections per pass {corrections}; final hyperplane {linear_accuracy(Phi_dr, y_dr, w, w0):.0%} correct; "
      f"w = {np.round(w, 3)}, w0 = {w0:.1f}")

for ax in axes:
    ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

### Question 10.1 (2 points)

- Run the perceptron on the separable and on the overlapping clouds (`visit_seed = 1`) and report, for both, the corrections in each pass, the training accuracy of the final hyperplane and that of the averaged perceptron. **(1 point)**
- Why does the algorithm stop on the separable data and never on the overlapping data? Apply the perceptron to the disc-and-ring data with the extra feature $|\vec x|^2$ (`add_radius_feature`) and show the resulting circle in the plane. **(1 point)**

In [ ]:
# Your code for Question 10.1

**Your answers for Question 10.1:**

## Logistic regression and gradient descent

Logistic regression (pp. 38–46): $P(y \mid \vec x) = e^{y a} / (e^{a} + e^{-a})$ with the score $a = \vec w\cdot\vec x + w_0$. The **energy** $E$, minus the log-likelihood of the training labels, is minimised by **gradient descent**; its gradient is the **error times the input** ($\tanh a_n$ is the expected label):

$$E(\vec w, w_0) = -\sum_{n=1}^N \log P(y_n \mid \vec x_n) = \sum_{n=1}^N \left[-y_n a_n + \log\left(e^{a_n} + e^{-a_n}\right)\right], \qquad \frac{\partial E}{\partial \vec w} = -\sum_{n=1}^N (y_n - \tanh a_n)\,\vec x_n, \qquad \frac{\partial E}{\partial w_0} = -\sum_{n=1}^N (y_n - \tanh a_n).$$

The cell fits the overlapping clouds and reproduces Figure 13 of the notes: the contours of $E(w_1, w_2)$ at the optimal $w_0$ and gradient descent from $\vec w = (-2.2, 5)$. `finite_difference_gradient(f, theta)` returns the central-difference gradient of `f` at `theta` (Question 10.3).

In [ ]:
def logistic_energy(w, w0, X, y) -> float:
    """E(w, w0) = -sum_n log P(y_n | x_n) = sum_n [-y_n a_n + log(e^{a_n} + e^{-a_n})], with a_n = w.x_n + w0."""
    a = X @ w + w0
    return float(np.sum(-y * a + np.logaddexp(a, -a)))


def logistic_gradient(w, w0, X, y):
    """dE/dw = -sum_n (y_n - tanh a_n) x_n and dE/dw0 = -sum_n (y_n - tanh a_n): the error times the input."""
    error = y - np.tanh(X @ w + w0)
    return -(error @ X), -float(error.sum())


def gradient_descent(X, y, w, w0, rate: float = 0.002, n_steps: int = 2000, fix_w0: bool = False):
    """(w, w0) <- (w, w0) - rate * grad E, repeated n_steps times.

    Returns the arrays of w (n_steps + 1 rows), of w0 and of E along the path (the first entry is the start).
    With fix_w0=True only w moves and w0 stays at its initial value (the energy slice of the notes' Figure 13).
    """
    w, w0 = np.array(w, dtype=float), float(w0)
    ws, w0s, energies = [w.copy()], [w0], [logistic_energy(w, w0, X, y)]
    for _ in range(n_steps):
        grad_w, grad_w0 = logistic_gradient(w, w0, X, y)
        w = w - rate * grad_w
        if not fix_w0:
            w0 = w0 - rate * grad_w0
        ws.append(w.copy())
        w0s.append(w0)
        energies.append(logistic_energy(w, w0, X, y))
    return np.array(ws), np.array(w0s), np.array(energies)


def probability_positive(w, w0, X):
    """P(y = 1 | x) = sigma(2a) = (1 + tanh a) / 2, with a = w.x + w0."""
    return 0.5 * (1 + np.tanh(X @ w + w0))


# the optimum (the teal star): a long descent on (w, w0) from zero
ws_fit, w0s_fit, Es_fit = gradient_descent(X_ovl, y_ovl, w=[0.0, 0.0], w0=0.0, rate=0.002, n_steps=4000)
w_star, w0_star = ws_fit[-1], w0s_fit[-1]
grad_norm = np.linalg.norm(np.append(*logistic_gradient(w_star, w0_star, X_ovl, y_ovl)))
print(f"fitted w* = {np.round(w_star, 3)}, w0* = {w0_star:.3f}  (|gradient| at the end {grad_norm:.1e}); "
      f"E at the optimum {Es_fit[-1]:.2f}; training accuracy {linear_accuracy(X_ovl, y_ovl, w_star, w0_star):.1%}")
print(f"in the notes' form sigma(w.x + w0), with the factor 2 absorbed into the weights: 2w* = {np.round(2 * w_star, 3)}, 2w0* = {2 * w0_star:.3f}")

# the descent of the figure: from w_start, with w0 held at w0*
w_start = np.array([-2.2, 5.0])
ws_path, w0s_path, Es_path = gradient_descent(X_ovl, y_ovl, w=w_start, w0=w0_star, rate=0.002, n_steps=1000, fix_w0=True)
print(f"gradient descent from w = {w_start} (w0 = w0*): E = {Es_path[0]:.1f} at the start, {Es_path[-1]:.2f} after {len(Es_path) - 1} steps; "
      f"E rises at some step: {bool(np.any(np.diff(Es_path) > 0))}")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))

# left: the data, the background P(y = 1 | x) and the hyperplane where it is 1/2
ax = axes[0]
x0, x1 = X_ovl[:, 0].min() - 0.7, X_ovl[:, 0].max() + 0.7
y0, y1 = X_ovl[:, 1].min() - 0.7, X_ovl[:, 1].max() + 0.7
gx, gy = np.meshgrid(np.linspace(x0, x1, 300), np.linspace(y0, y1, 300))
P_grid = probability_positive(w_star, w0_star, np.stack([gx.ravel(), gy.ravel()], 1)).reshape(gx.shape)
ax.imshow(P_grid, extent=(x0, x1, y0, y1), origin="lower", cmap="bwr", vmin=0, vmax=1, alpha=0.45)
plot_points(X_ovl, y_ovl, ax)
plot_hyperplane(w_star, w0_star, ax, "black", lw=2, label=r"$P(y = 1 \mid \vec x) = 1/2$")
ax.set_title(r"logistic regression; background: $P(y = 1 \mid \vec x)$ from blue (0) to red (1)")
ax.legend(loc="upper left", fontsize=8)

# right: the contours of E(w1, w2) at w0 = w0*, the descent path and the minimum
ax = axes[1]
lo = np.minimum(ws_path.min(0), w_star) - 1.5
hi = np.maximum(ws_path.max(0), w_star) + 1.5
g1, g2 = np.meshgrid(np.linspace(lo[0], hi[0], 121), np.linspace(lo[1], hi[1], 121))
A = g1.ravel()[:, None] * X_ovl[:, 0][None, :] + g2.ravel()[:, None] * X_ovl[:, 1][None, :] + w0_star
E_grid = np.sum(-y_ovl * A + np.logaddexp(A, -A), axis=1).reshape(g1.shape)
ax.contour(g1, g2, E_grid, levels=E_grid.min() + np.geomspace(1, E_grid.max() - E_grid.min(), 14), colors=[GREY], linewidths=0.8)
ax.plot(ws_path[:, 0], ws_path[:, 1], color=ORANGE, lw=1.5, label="gradient descent")
ax.plot(ws_path[:401:40, 0], ws_path[:401:40, 1], "o", color=ORANGE, ms=4, label="every 40 steps over the first 400")
ax.plot(w_star[0], w_star[1], "*", color=TEAL, ms=16, label="global minimum")
ax.set_xlabel("$w_1$")
ax.set_ylabel("$w_2$")
ax.set_title(r"the energy $E(w_1, w_2)$ with $w_0$ fixed at its optimum")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
# the same descent on the separable clouds
ws_sep, w0s_sep, Es_sep = gradient_descent(X_sep, y_sep, w=[0.0, 0.0], w0=0.0, rate=0.0005, n_steps=5000)
norms_sep = np.linalg.norm(ws_sep, axis=1)
print(f"separable clouds: |w| after 10, 100, 1000 and 5000 iterations: {[round(float(norms_sep[i]), 2) for i in (10, 100, 1000, 5000)]}; "
      f"E = {Es_sep[0]:.1f} at the start, {Es_sep[-1]:.4f} after 5000 iterations; E rises at some step: {bool(np.any(np.diff(Es_sep) > 0))}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(norms_sep, color=TEAL)
axes[0].set_xscale("log")
axes[0].set_xlabel("iteration $t$")
axes[0].set_ylabel(r"$|\vec w|$")
axes[0].set_title("separable clouds: length of the weight vector")
axes[1].plot(Es_sep, color=TEAL)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("iteration $t$")
axes[1].set_ylabel("energy $E$")
axes[1].set_title("separable clouds: energy")
fig.tight_layout()
plt.show()

### Question 10.2 (2 points)

- Report the fitted $(\vec w, w_0)$ on the overlapping clouds and the energy $E$ at the start $\vec w = (-2.2, 5)$ and at the end of the descent, and show the path on the contour plot. Why can $E$ never rise along the path for a small enough learning rate? **(1 point)**
- On the separable clouds, what happens to $|\vec w|$ as the descent goes on, and why? What keeps the optimum finite (Lecture 8)? **(1 point)**

In [ ]:
# Your code for Question 10.2

**Your answers for Question 10.2:**

## A multi-layer perceptron trained by backpropagation

The example network of the notes (pp. 48–70) stacks soft perceptrons,

$$y = \sigma(\vec w\cdot\vec z + w_0), \qquad z_i = \sigma(\vec\Omega^i\cdot\vec x + \Omega^i_0), \quad i = 1, 2, 3, \qquad \sigma(u) = \frac{1}{1 + e^{-u}},$$

each hidden unit $z_i$ having its own hyperplane. With the label $t = (y_{\text{label}} + 1)/2$ the energy is the cross-entropy $E = -t\log y - (1 - t)\log(1 - y)$ summed over the examples; **backpropagation** gives every derivative from the forward pass, with $u_i = \vec\Omega^i\cdot\vec x + \Omega^i_0$:

$$\frac{\partial E}{\partial w_i} = (y - t)\, z_i, \qquad \frac{\partial E}{\partial w_0} = y - t, \qquad \frac{\partial E}{\partial \vec\Omega^i} = \delta_i\, \vec x, \qquad \frac{\partial E}{\partial \Omega^i_0} = \delta_i = \frac{\partial E}{\partial u_i} = (y - t)\, w_i\, z_i (1 - z_i).$$

**Steepest descent** averages the gradient over all the examples, **SGD** follows one random example, **minibatch** SGD a random subset; **momentum** adds 0.9 times the previous step. `pack`/`unpack` map the weights to one vector of $4H + 1$ numbers; `mlp_energy` is the total energy.

In [ ]:
def sigmoid(u):
    """sigma(u) = 1 / (1 + e^{-u}), computed as (1 + tanh(u/2)) / 2 so that large |u| does not overflow."""
    return 0.5 * (1.0 + np.tanh(0.5 * u))


def cross_entropy(y_net, t, tiny: float = 1e-12) -> float:
    """Sum over the examples of -t log y - (1 - t) log(1 - y); y is clipped to [tiny, 1 - tiny] so that log 0 never occurs."""
    y_net = np.clip(y_net, tiny, 1 - tiny)
    return float(np.sum(-t * np.log(y_net) - (1 - t) * np.log(1 - y_net)))


def init_mlp(n_hidden: int = 3, seed: int = 0, scale: float = 1.0):
    """Random weights (Omega, Omega0, w, w0) ~ N(0, scale^2): Omega is n_hidden x 2, Omega0 and w have n_hidden entries."""
    rng = np.random.default_rng(seed)
    return (rng.normal(scale=scale, size=(n_hidden, 2)), rng.normal(scale=scale, size=n_hidden),
            rng.normal(scale=scale, size=n_hidden), float(rng.normal(scale=scale)))


def mlp_forward(X, Omega, Omega0, w, w0):
    """z_i = sigma(Omega^i . x + Omega^i_0) for every hidden unit and y = sigma(w . z + w0). Returns z (N x n_hidden) and y (N)."""
    z = sigmoid(X @ Omega.T + Omega0)
    return z, sigmoid(z @ w + w0)


def mlp_backward(X, t, Omega, Omega0, w, w0):
    """Backpropagation for the total energy of the examples (X, t), exactly as in the notes:
         dE/du = y - t;   dE/dw_i = (y - t) z_i;   dE/dw0 = y - t;
         delta_i = dE/du_i = (y - t) w_i z_i (1 - z_i);   dE/dOmega^i = delta_i x;   dE/dOmega^i_0 = delta_i,
    each summed over the examples. Returns (dOmega, dOmega0, dw, dw0) with the shapes of the weights."""
    z, y_net = mlp_forward(X, Omega, Omega0, w, w0)
    error = y_net - t                                    # dE/du: one number per example
    delta = error[:, None] * w[None, :] * z * (1 - z)    # dE/du_i: N x n_hidden
    return delta.T @ X, delta.sum(0), error @ z, float(error.sum())


def mlp_energy(X, t, Omega, Omega0, w, w0) -> float:
    """The total cross-entropy energy of the examples (X, t)."""
    return cross_entropy(mlp_forward(X, Omega, Omega0, w, w0)[1], t)


def pack(Omega, Omega0, w, w0):
    """All the weights in one vector: Omega row by row, then Omega0, then w, then w0 (4 n_hidden + 1 numbers)."""
    return np.concatenate([np.ravel(Omega), Omega0, w, [w0]])


def unpack(theta, n_hidden: int):
    """The inverse of pack: returns (Omega, Omega0, w, w0)."""
    theta = np.asarray(theta, dtype=float)
    return (theta[:2 * n_hidden].reshape(n_hidden, 2), theta[2 * n_hidden:3 * n_hidden],
            theta[3 * n_hidden:4 * n_hidden], float(theta[4 * n_hidden]))


def permute_hidden(params, order):
    """The same network with its hidden units in a new order: the rows of Omega, the entries of Omega0 and of w move together."""
    Omega, Omega0, w, w0 = params
    order = list(order)
    return Omega[order], Omega0[order], w[order], w0


def train_mlp(X, y, n_hidden: int = 3, mode: str = "steepest", batch_size: int = 16, rate: float = 0.5, momentum: float = 0.9,
              n_passes: int = 400, seed: int = 0, init=None):
    """Train the MLP on labels y in {-1, +1} (t = (y + 1)/2) by gradient descent on the cross-entropy energy.

    mode = "steepest":  one step per pass, with the gradient averaged over all N examples (steepest descent, SD);
    mode = "sgd":       N steps per pass, one example at a time in a random order (stochastic gradient descent);
    mode = "minibatch": steps on random minibatches of batch_size examples (the last minibatch of a pass may be smaller).
    Every step adds momentum times the previous step: v <- momentum v - rate * (mean gradient over the batch); weights <- weights + v.
    init: starting weights (Omega, Omega0, w, w0); by default init_mlp(n_hidden, seed).
    Returns the trained (Omega, Omega0, w, w0) and the total energy E on all the data before training and after every pass.
    """
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1")
    rng = np.random.default_rng(seed)
    N = len(X)
    t = (np.asarray(y, dtype=float) + 1) / 2
    Omega, Omega0, w, w0 = init if init is not None else init_mlp(n_hidden, seed)
    Omega, Omega0, w, w0 = Omega.copy(), Omega0.copy(), w.copy(), float(w0)
    v_Omega, v_Omega0, v_w, v_w0 = np.zeros_like(Omega), np.zeros_like(Omega0), np.zeros_like(w), 0.0
    energies = [mlp_energy(X, t, Omega, Omega0, w, w0)]
    for _ in range(n_passes):
        if mode == "steepest":
            batches = [np.arange(N)]
        elif mode == "sgd":
            batches = rng.permutation(N)[:, None]
        elif mode == "minibatch":
            perm = rng.permutation(N)
            batches = [perm[i:i + batch_size] for i in range(0, N, batch_size)]
        else:
            raise ValueError(mode)
        for b in batches:
            g_Omega, g_Omega0, g_w, g_w0 = mlp_backward(X[b], t[b], Omega, Omega0, w, w0)
            m = len(b)
            v_Omega = momentum * v_Omega - rate * g_Omega / m
            v_Omega0 = momentum * v_Omega0 - rate * g_Omega0 / m
            v_w = momentum * v_w - rate * g_w / m
            v_w0 = momentum * v_w0 - rate * g_w0 / m
            Omega, Omega0, w, w0 = Omega + v_Omega, Omega0 + v_Omega0, w + v_w, w0 + v_w0
        energies.append(mlp_energy(X, t, Omega, Omega0, w, w0))
    return (Omega, Omega0, w, w0), np.array(energies)


def plot_mlp_panels(X, y, params, w_lr, w0_lr):
    """The three panels of the notes' Figure 15: the logistic hyperplane (olive), the hidden hyperplanes (purple),
    and the output of the MLP with the teal curve where it is 1/2."""
    Omega, Omega0, w, w0 = params
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
    for ax in axes:
        plot_points(X, y, ax)
    plot_hyperplane(w_lr, w0_lr, axes[0], OLIVE, lw=2.5, label="logistic regression")
    axes[0].set_title("logistic regression: one hyperplane")
    for i in range(len(w)):
        plot_hyperplane(Omega[i], Omega0[i], axes[1], PURPLE, lw=2, label="hidden hyperplanes" if i == 0 else None)
        label_hyperplane(Omega[i], Omega0[i], axes[1], str(i + 1), PURPLE)
    axes[1].set_title(rf"the {len(w)} hidden hyperplanes $\vec\Omega^i\cdot\vec x + \Omega^i_0 = 0$")
    x0, x1 = axes[2].get_xlim()
    y0, y1 = axes[2].get_ylim()
    gx, gy = np.meshgrid(np.linspace(x0, x1, 300), np.linspace(y0, y1, 300))
    output = mlp_forward(np.stack([gx.ravel(), gy.ravel()], 1), *params)[1].reshape(gx.shape)
    axes[2].imshow(output, extent=(x0, x1, y0, y1), origin="lower", cmap="bwr", vmin=0, vmax=1, alpha=0.35)
    plot_level_set(lambda points: mlp_forward(points, *params)[1], axes[2], TEAL, level=0.5, label="output $y = 1/2$")
    axes[2].set_title("the MLP: output $y$ (red above 1/2, blue below)")
    for ax in axes:
        ax.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    plt.show()

In [ ]:
t_dr = (y_dr + 1) / 2

# steepest descent with momentum 0.9 from random weights (the training of the notes, p. 53)
params_A, energies_A = train_mlp(X_dr, y_dr, n_hidden=3, mode="steepest", rate=0.5, momentum=0.9, n_passes=10000, seed=0)
Omega_A, Omega0_A, w_A, w0_A = params_A
z_dr, y_net_dr = mlp_forward(X_dr, *params_A)
mlp_accuracy = float(np.mean((y_net_dr > 0.5) == (t_dr == 1)))

# logistic regression on the same data
ws_lr, w0s_lr, Es_lr = gradient_descent(X_dr, y_dr, w=[0.0, 0.0], w0=0.0, rate=0.002, n_steps=2000)
w_lr, w0_lr = ws_lr[-1], w0s_lr[-1]

print(f"MLP with 3 hidden units: E = {energies_A[0]:.2f} at the random start, {energies_A[-1]:.3f} after {len(energies_A) - 1} steps of "
      f"steepest descent with momentum 0.9; training accuracy {mlp_accuracy:.1%}")
print(f"logistic regression on the same data: E = {Es_lr[-1]:.1f}, training accuracy {linear_accuracy(X_dr, y_dr, w_lr, w0_lr):.1%}")
print("\ntrained weights:   Omega^i (input weights)    Omega^i_0 (bias)    w_i (output weight)")
for i in range(len(w_A)):
    print(f"  hidden unit {i + 1}:   ({Omega_A[i, 0]:7.2f}, {Omega_A[i, 1]:7.2f})      {Omega0_A[i]:8.2f}        {w_A[i]:8.2f}")
print(f"  output bias w0 = {w0_A:.2f}")

plt.figure(figsize=(5.5, 3.2))
plt.plot(energies_A, color=TEAL)
plt.yscale("log")
plt.xlabel("step of steepest descent (one pass through the data each)")
plt.ylabel("energy $E$")
plt.title("MLP: energy during training (momentum 0.9)")
plt.show()

plot_mlp_panels(X_dr, y_dr, params_A, w_lr, w0_lr)

In [ ]:
# steepest descent, SGD and minibatches from the same random weights, without momentum (the notes' Figure 18, p. 67)
init_B = init_mlp(n_hidden=3, seed=0)
n_passes_compare = 400
rules = {"steepest descent (all 240 examples per step)": ("steepest", 2.0, "black"),
         "minibatches of 16 examples": ("minibatch", 0.6, TEAL),
         "SGD, one example per step": ("sgd", 0.2, ORANGE)}
plt.figure(figsize=(7.5, 4.2))
for name, (mode, rate, colour) in rules.items():
    _, energies = train_mlp(X_dr, y_dr, mode=mode, batch_size=16, rate=rate, momentum=0.0, n_passes=n_passes_compare, seed=0, init=init_B)
    evaluations = np.arange(len(energies)) * len(X_dr)            # per-example gradient evaluations after every pass
    plt.plot(evaluations, energies / len(X_dr), color=colour, label=f"{name}, rate {rate}")
    steps = {"steepest": 1, "minibatch": int(np.ceil(len(X_dr) / 16)), "sgd": len(X_dr)}[mode] * n_passes_compare
    print(f"{name:47s}: energy per example {energies[-1] / len(X_dr):.4f} after {n_passes_compare} passes "
          f"= {evaluations[-1]} gradient evaluations, {steps} weight updates")
plt.yscale("log")
plt.xlabel("number of gradient evaluations (examples processed)")
plt.ylabel("energy per example $E/N$")
plt.title("the three update rules from the same start, no momentum")
plt.legend(fontsize=8)
plt.show()

In [ ]:
# the permutation symmetry: hidden units 1 and 2 swapped, with their output weights (the notes' Figure 19, p. 70)
params_B = permute_hidden(params_A, order=[1, 0, 2])
E_A, E_B = mlp_energy(X_dr, t_dr, *params_A), mlp_energy(X_dr, t_dr, *params_B)
print(f"E(A) = {E_A:.4f}, E(B) = {E_B:.4f}, |E(A) - E(B)| = {abs(E_A - E_B):.1e}")
theta_A, theta_B = pack(*params_A), pack(*params_B)
print(f"largest difference between the two weight vectors: {np.abs(theta_A - theta_B).max():.2f}")

s_values = np.linspace(0, 1, 101)
E_line = np.array([mlp_energy(X_dr, t_dr, *unpack((1 - s) * theta_A + s * theta_B, len(w_A))) for s in s_values])
print(f"E at the midpoint of the straight line from A to B: {E_line[50]:.2f}")
plt.figure(figsize=(6.5, 3.5))
plt.plot(s_values, E_line, color=TEAL)
plt.plot([0, 1], [E_A, E_B], "o", color=TEAL)
plt.xlabel("position $s$ on the straight line from the weights A ($s = 0$) to B ($s = 1$)")
plt.ylabel("energy $E$")
plt.title("the energy along the segment between two permuted copies of the trained network")
plt.show()

### Question 10.3 (3 points)

- Report $E$ at the start and at the end of training and the training accuracy of the MLP and of logistic regression, and show the three panels. Then check the backprop gradient at a random weight vector (`init_mlp`): report $\max_j |g_j^{\text{bp}} - g_j^{\text{fd}}| \,/\, \max_j |g_j^{\text{fd}}|$ over the 13 weights (`mlp_backward` against `finite_difference_gradient`). **(1 point)**
- Which hidden units $z_i$ are close to 1 inside the disc (`mlp_forward` on the disc points)? Then retrain with `n_hidden=6` (starter line below), show its three panels (`plot_mlp_panels`) and describe the positive region. **(1 point)**
- On the plot of the energy per example against the number of gradient evaluations, which of steepest descent, SGD and minibatches of 16 reaches a low energy with the least computation? Swapping hidden units 1 and 2 with their output weights leaves $E$ unchanged (`permute_hidden`): what does this imply about the shape of $E$ as a function of the weights? **(1 point)**

In [ ]:
# Your code for Question 10.3
# starter for the 6-unit retraining: params_6, energies_6 = train_mlp(X_dr, y_dr, n_hidden=6, mode="steepest", rate=0.5, momentum=0.9, n_passes=10000, seed=0)

**Your answers for Question 10.3:**

## Softmax regression and a small convolutional network

The rest of the notebook works on $16 \times 16$ images of three shapes (horizontal bar, vertical bar, disc; 3000 training, 1000 test). Writing $x_i$ for the **total input** of output unit $i$, the **softmax** gives the probability that image $I$ shows class $i$ (pp. 95–96),

$$f(x_i) = \frac{\exp(x_i)}{\sum_j \exp(x_j)} = P(y = i \mid I, \omega),$$

with weights $\omega$; training maximises $\log f(x_t)$ of the correct label $t$ over the training images, maximum likelihood for the regression model $P(y \mid I, \omega)$. The **loss** $-\log f(x_t) = -\log P(y_{\text{true}} \mid I, \omega)$ of an image is the cross-entropy with several classes.

**Softmax regression** (`LayerNetwork([256, 3])`, one linear layer), **the MLP** (`LayerNetwork([256, 64, 32, 3])`, two layers of rectified linear units) and **a small convolutional network** (`ConvNetwork`: 8 rectified $5 \times 5$ filters, $2 \times 2$ max pooling, a linear layer from the 288 pooled responses) are trained by the same minibatch SGD (minibatches of 50, rate 0.05, momentum 0.9, 20 passes); keep other rates below about 0.3, or every rectified unit switches off and the loss sticks at $\log 3$. `train_model` records the test accuracy every pass, or every `evaluate_every` updates, when given `evaluate=(X_test, y_test)`; `show_filters` draws the filters.

In [ ]:
CLASS_NAMES = ["horizontal bar", "vertical bar", "disc"]


def shape_images(n: int, size: int = 16, noise: float = 0.1, seed: int = 0):
    """n grey-level images (size x size, flattened to vectors of size^2 numbers in [0, 1]) and their labels 0, 1, 2.

    Background 0.2, shape 0.8, plus Gaussian pixel noise of standard deviation `noise`, clipped to [0, 1]:
      0: a horizontal bar, 2 or 3 pixels thick and 6 to 10 long, at a random position;  1: a vertical bar;
      2: a disc of radius 2 to 3.5 at a random position.
    """
    if size < 12:
        raise ValueError("size must be at least 12 so that the shapes fit")
    rng = np.random.default_rng(seed)
    labels = rng.integers(3, size=n)
    X = np.full((n, size, size), 0.2)
    rows, cols = np.mgrid[0:size, 0:size]
    for k in range(n):
        if labels[k] == 0:
            length, thickness = rng.integers(6, 11), rng.integers(2, 4)
            r0, c0 = rng.integers(0, size - thickness + 1), rng.integers(0, size - length + 1)
            X[k, r0:r0 + thickness, c0:c0 + length] = 0.8
        elif labels[k] == 1:
            length, thickness = rng.integers(6, 11), rng.integers(2, 4)
            r0, c0 = rng.integers(0, size - length + 1), rng.integers(0, size - thickness + 1)
            X[k, r0:r0 + length, c0:c0 + thickness] = 0.8
        else:
            radius = rng.uniform(2.0, 3.5)
            cy, cx = rng.uniform(radius, size - 1 - radius, size=2)
            X[k][(rows - cy) ** 2 + (cols - cx) ** 2 <= radius**2] = 0.8
    X += rng.normal(scale=noise, size=X.shape)
    return np.clip(X, 0, 1).reshape(n, -1), labels


def show_images(X, labels, predictions=None, title: str = "", size: int = 16, n_show: int = 8):
    """A row of images (vmin 0, vmax 1) titled with their labels or, if given, their predicted labels: teal if correct, orange if wrong."""
    fig, axes = plt.subplots(1, n_show, figsize=(1.6 * n_show, 2.2))
    for k, ax in enumerate(axes):
        ax.imshow(X[k].reshape(size, size), cmap="gray", vmin=0, vmax=1)
        if predictions is None:
            ax.set_title(CLASS_NAMES[labels[k]], fontsize=8)
        else:
            ax.set_title(CLASS_NAMES[predictions[k]], fontsize=8, color=TEAL if predictions[k] == labels[k] else ORANGE)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(title)
    plt.show()


def show_filters(W, title: str = ""):
    """The filters W (n_filters x size x size) as images on a grey scale symmetric about zero (black negative, white positive)."""
    m = float(np.abs(W).max()) or 1.0
    fig, axes = plt.subplots(1, len(W), figsize=(1.5 * len(W), 1.9))
    for k, ax in enumerate(axes):
        ax.imshow(W[k], cmap="gray", vmin=-m, vmax=m)
        ax.set_title(f"filter {k + 1}", fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(title)
    plt.show()


def softmax(u):
    """The softmax of every row: exp(x_i) / sum_j exp(x_j), computed after subtracting the largest total input of the row."""
    e = np.exp(u - u.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def losses_of(P, labels, tiny: float = 1e-12):
    """The loss -log P(y_true | x) of every example; P is clipped to [tiny, 1] so that log 0 never occurs."""
    return -np.log(np.clip(P[np.arange(len(labels)), labels], tiny, 1))


class LayerNetwork:
    """Fully connected layers sizes[0] -> sizes[1] -> ... -> sizes[-1], a ReLU after every layer except the last, which is a
    softmax over the classes. sizes = [256, 3] is softmax regression; sizes = [256, 64, 32, 3] is the MLP.
    The weights W_l are white Gaussian noise of variance 2 / (number of inputs of the layer) and the biases b_l = 0, drawn with the
    given seed."""

    def __init__(self, sizes, seed: int = 0):
        rng = np.random.default_rng(seed)
        self.sizes = list(sizes)
        self.params = []                                       # [W_1, b_1, W_2, b_2, ...]
        for a, b in zip(sizes[:-1], sizes[1:]):
            self.params += [rng.normal(scale=np.sqrt(2 / a), size=(a, b)), np.zeros(b)]
        self.layer_names = ["image"] + [f"hidden {l + 1}" for l in range(len(sizes) - 2)] + ["output probabilities"]

    @property
    def n_weights(self) -> int:
        return sum(p.size for p in self.params)

    def forward(self, X):
        """The activities of every layer, [x, h_1, ..., P(class | x)]: h_l = ReLU(W_l h_{l-1} + b_l), the last layer a softmax."""
        h = [X]
        n_layers = len(self.params) // 2
        for l in range(n_layers):
            u = h[-1] @ self.params[2 * l] + self.params[2 * l + 1]
            h.append(np.maximum(u, 0) if l < n_layers - 1 else softmax(u))
        return h

    def backward(self, X, labels):
        """Backpropagation. Returns (grads, grad_x, losses): grads = [dL/dW_1, dL/db_1, ...] for the loss summed over the
        examples, grad_x = dL/dx for every example (the gradient of its own loss with respect to its own pixels), and the
        loss of every example."""
        h = self.forward(X)
        n_layers = len(self.params) // 2
        delta = h[-1].copy()                                   # dL/du of the output layer: P - [i = t], prediction minus target (p. 96, with the sign of the loss -log f(x_t))
        delta[np.arange(len(labels)), labels] -= 1
        grads = [None] * len(self.params)
        for l in range(n_layers - 1, -1, -1):
            grads[2 * l], grads[2 * l + 1] = h[l].T @ delta, delta.sum(0)
            if l > 0:
                delta = (delta @ self.params[2 * l].T) * (h[l] > 0)   # through the ReLU: zero where the unit is off
        return grads, delta @ self.params[0].T, losses_of(h[-1], labels)


class ConvNetwork:
    """One convolutional layer (n_filters filters of filter_size x filter_size, applied at every position, ReLU), 2 x 2 max
    pooling, and a linear layer from the pooled maps to the classes, followed by the softmax. Images are size x size."""

    def __init__(self, size: int = 16, n_filters: int = 8, filter_size: int = 5, n_classes: int = 3, seed: int = 0):
        rng = np.random.default_rng(seed)
        self.size, self.n_filters, self.filter_size = size, n_filters, filter_size
        self.map_size = size - filter_size + 1                 # 12 for 16 x 16 images and 5 x 5 filters
        if self.map_size % 2:
            raise ValueError("size - filter_size + 1 must be even for 2 x 2 pooling")
        self.pool_size = self.map_size // 2                    # 6
        n_pooled = n_filters * self.pool_size**2               # 288
        self.params = [rng.normal(scale=np.sqrt(2 / filter_size**2), size=(n_filters, filter_size, filter_size)), np.zeros(n_filters),
                       rng.normal(scale=np.sqrt(2 / n_pooled), size=(n_pooled, n_classes)), np.zeros(n_classes)]
        self.layer_names = ["image", "feature maps", "pooled maps", "output probabilities"]

    @property
    def n_weights(self) -> int:
        return sum(p.size for p in self.params)

    def _patches(self, X):
        """Every filter_size x filter_size window of every image: N x (map_size^2) x (filter_size^2)."""
        images = X.reshape(len(X), self.size, self.size)
        windows = sliding_window_view(images, (self.filter_size, self.filter_size), axis=(1, 2))
        return windows.reshape(len(X), self.map_size**2, self.filter_size**2)

    def _forward_all(self, X):
        W1, b1, W2, b2 = self.params
        N, q, f = len(X), self.pool_size, self.n_filters
        patches = self._patches(X)
        total_input = patches @ W1.reshape(f, -1).T + b1                 # N x positions x filters: sum_uv W_uv I_{i+u,j+v} + w0
        maps = np.maximum(total_input, 0)                                # the feature maps after the ReLU
        windows = maps.reshape(N, q, 2, q, 2, f)                         # 2 x 2 windows of each map
        pooled = windows.max(axis=(2, 4))                                # N x q x q x filters
        flat = pooled.reshape(N, -1)
        P = softmax(flat @ W2 + b2)
        return patches, total_input, maps, windows, pooled, flat, P

    def forward(self, X):
        """The activities of every layer: [x, feature maps, pooled maps, P(class | x)], each as one vector per image."""
        patches, total_input, maps, windows, pooled, flat, P = self._forward_all(X)
        return [X, maps.reshape(len(X), -1), flat, P]

    def backward(self, X, labels):
        """Backpropagation through the linear layer, the max pooling (to the position of the maximum of each window), the ReLU
        and the convolution. Returns (grads, grad_x, losses) like LayerNetwork.backward, grads = [dW1, db1, dW2, db2]."""
        W1, b1, W2, b2 = self.params
        N, m, q, f, fs = len(X), self.map_size, self.pool_size, self.n_filters, self.filter_size
        patches, total_input, maps, windows, pooled, flat, P = self._forward_all(X)
        delta = P.copy()
        delta[np.arange(N), labels] -= 1                                 # P - [i = t], prediction minus target, as in LayerNetwork.backward
        dW2, db2 = flat.T @ delta, delta.sum(0)
        d_pooled = (delta @ W2.T).reshape(N, q, 1, q, 1, f)
        at_max = windows == pooled.reshape(N, q, 1, q, 1, f)             # the position that gave the maximum of each window
        d_maps = (at_max * d_pooled).reshape(N, m * m, f) * (total_input > 0)
        dW1 = np.einsum("npk,npf->fk", patches, d_maps).reshape(f, fs, fs)
        db1 = d_maps.sum((0, 1))
        d_patches = (d_maps @ W1.reshape(f, -1)).reshape(N, m, m, fs, fs)
        grad_x = np.zeros((N, self.size, self.size))
        for u in range(fs):
            for v in range(fs):
                grad_x[:, u:u + m, v:v + m] += d_patches[:, :, :, u, v]  # every window adds its share to its pixels
        return [dW1, db1, dW2, db2], grad_x.reshape(N, -1), losses_of(P, labels)


def predict(model, X):
    return model.forward(X)[-1].argmax(1)


def accuracy(model, X, labels) -> float:
    return float(np.mean(predict(model, X) == labels))


def train_model(model, X, labels, n_passes: int = 20, batch_size: int = 50, rate: float = 0.05, momentum: float = 0.9, seed: int = 0,
                adversarial_eps=None, evaluate=None, evaluate_every=None):
    """Minibatch SGD with momentum on the mean loss of the minibatch, for n_passes passes through the data; the weights of the
    model are updated in place. With adversarial_eps, the FGSM examples of every minibatch (made with the current weights, at
    that eps) are added to the minibatch: adversarial training. evaluate = (X_eval, y_eval) records the accuracy on these
    examples before training and after every pass, or after every evaluate_every weight updates and at the end of training.
    Returns history = {"loss": mean training loss of every pass, "passes": passes completed at each evaluation, "accuracy": ...}."""
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1")
    rng = np.random.default_rng(seed)
    velocity = [np.zeros_like(p) for p in model.params]
    batches_per_pass = int(np.ceil(len(X) / batch_size))
    history = {"loss": [], "passes": [], "accuracy": []}

    def record(n_updates):
        if evaluate is not None:
            history["passes"].append(n_updates / batches_per_pass)
            history["accuracy"].append(accuracy(model, *evaluate))

    n_updates = 0
    record(n_updates)
    for _ in range(n_passes):
        perm = rng.permutation(len(X))
        total, count = 0.0, 0
        for i in range(0, len(X), batch_size):
            b = perm[i:i + batch_size]
            X_b, y_b = X[b], labels[b]
            if adversarial_eps:
                X_b = np.concatenate([X_b, fgsm(model, X_b, y_b, adversarial_eps)])
                y_b = np.concatenate([y_b, y_b])
            grads, _, losses = model.backward(X_b, y_b)
            total += losses.sum()
            count += len(y_b)
            for p, v, grad in zip(model.params, velocity, grads):
                v *= momentum
                v -= rate * grad / len(y_b)
                p += v
            n_updates += 1
            if evaluate_every and n_updates % evaluate_every == 0:
                record(n_updates)
        history["loss"].append(total / count)
        if not evaluate_every:
            record(n_updates)
    if evaluate_every and n_updates % evaluate_every:
        record(n_updates)
    return {key: np.array(values) for key, values in history.items()}


SIZE = 16
X_train, y_train = shape_images(3000, size=SIZE, seed=1)
X_test, y_test = shape_images(1000, size=SIZE, seed=2)
show_images(X_test, y_test, title="test images with their labels", size=SIZE)

In [ ]:
# the three models are created here with seeded random weights, evaluated untrained, then trained in place;
# re-running this cell therefore starts again from the same untrained models and reproduces the same numbers
softmax_reg = LayerNetwork([SIZE * SIZE, 3], seed=0)
mlp = LayerNetwork([SIZE * SIZE, 64, 32, 3], seed=0)
conv_net = ConvNetwork(size=SIZE, n_filters=8, filter_size=5, n_classes=3, seed=0)
models = {"softmax regression": softmax_reg, "MLP (64, 32 hidden units)": mlp, "convolutional network": conv_net}

print("test accuracy of the untrained models (seeded random weights):")
for name, model in models.items():
    print(f"  {name:28s} {accuracy(model, X_test, y_test):.1%}")

histories = {}
for name, model in models.items():
    histories[name] = train_model(model, X_train, y_train, n_passes=20, batch_size=50, rate=0.05, momentum=0.9, seed=0)
print("\nafter 20 passes of minibatch SGD (minibatches of 50, rate 0.05, momentum 0.9):")
for name, model in models.items():
    print(f"  {name:28s} mean training loss {histories[name]['loss'][0]:.3f} in the first pass and {histories[name]['loss'][-1]:.4f} in the last; "
          f"training accuracy {accuracy(model, X_train, y_train):.1%}, test accuracy {accuracy(model, X_test, y_test):.1%}; "
          f"{model.n_weights:,} weights")

plt.figure(figsize=(6.5, 3.8))
for (name, history), colour in zip(histories.items(), (OLIVE, PURPLE, TEAL)):
    plt.plot(np.arange(1, len(history["loss"]) + 1), history["loss"], "o-", ms=3, color=colour, label=name)
plt.yscale("log")
plt.xlabel("pass through the training data")
plt.ylabel("mean training loss $-\\log P(y_{\\rm true} \\mid I, \\omega)$")
plt.title("training loss of the three models")
plt.legend(fontsize=8)
plt.show()

show_images(X_test, y_test, predict(conv_net, X_test), title="labels predicted by the convolutional network (teal: correct, orange: wrong)", size=SIZE)
show_filters(conv_net.params[0], title="the 8 learnt filters of the convolutional layer (black: negative, white: positive weights)")

### Question 10.4 (3 points)

- Before training, what accuracy do you expect each of the three models to have on the three-class test set, and why? Evaluate the untrained models (printed above) and report the numbers. **(1 point)**
- Which loss function is minimised, and why is it the right loss for a classification problem? Report the test accuracy of softmax regression, the MLP and the convolutional network after training and show the learnt filters (`show_filters`). **(1 point)**
- Plot the test accuracy of softmax regression against the number of passes through the data, from a fraction of a pass to 30 passes (`train_model(LayerNetwork([256, 3]), X_train, y_train, n_passes=30, evaluate=(X_test, y_test), evaluate_every=10)`). Why does the convolutional network reach a higher accuracy with far fewer weights than the MLP (the counts are printed above)? **(1 point)**

In [ ]:
# Your code for Question 10.4

**Your answers for Question 10.4:**

## Adversarial examples

An **adversarial example** is a perturbation $r$ of an image $x$, with $|r_i| \le \varepsilon$ for every pixel (the set $S$), that raises the loss of the true label as much as possible (Lecture 11, pp. 10–21).

**The linear argument** (pp. 14–15): a linear unit's response to $x + \eta$ changes by $w^\top\eta$, at most $\varepsilon\|w\|_1 = \varepsilon m n$ (at $\eta_i = \varepsilon\,\text{sign}(w_i)$; $n$ inputs of mean magnitude $m$), while random signs $\eta_i = \pm\varepsilon$ change it by $\varepsilon\|w\|_2$ on average; for the network, $\text{loss}(f(x + r)) \approx \text{loss}(f(x)) + g^\top r$, $g = \nabla_x\,\text{loss}$.

**The attacks** (pp. 17–21): FGSM takes one step of size $\varepsilon$ along the sign of the gradient; PGD takes several steps of size $\alpha$, each projected ($\Pi$) back onto $x + S$ and $[0, 1]$; **random signs** $r_i = \pm\varepsilon$ are the control:

$$\text{FGSM:}\ r = \varepsilon\,\text{sign}\big(\nabla_x\,\text{loss}(f(x), y_{\text{true}}; \theta)\big), \qquad \text{PGD:}\ x_{t+1} = \Pi_{x + S}\big(x_t + \alpha\,\text{sign}(\nabla_x\,\text{loss}(f(x_t), y_{\text{true}}; \theta))\big), \quad x_0 = x .$$

**Min-max training** (p. 59) replaces the loss of each training image by its worst case, $\min_\theta \max_{r \in S} \text{loss}(f(x + r), y_{\text{true}}; \theta)$. The attacks below are on the MLP of the previous section (`mlp`); `relative_change` returns $\|h_l(x + r) - h_l(x)\| / \|h_l(x)\|$ for every layer $l$.

In [ ]:
def input_gradient(model, X, labels):
    """g = grad_x loss(f(x), y_true; theta) for every image."""
    return model.backward(X, labels)[1]


def fgsm(model, X, labels, eps: float):
    """x + eps sign(g), clipped to the pixel range [0, 1]."""
    return np.clip(X + eps * np.sign(input_gradient(model, X, labels)), 0, 1)


def pgd(model, X, labels, eps: float, alpha: float, n_steps: int = 5):
    """n_steps steps x <- Pi_{x+S}(x_t + alpha sign(g)), starting from x; the projection clips to [x - eps, x + eps] and to [0, 1]."""
    X_adv = X.copy()
    for _ in range(n_steps):
        X_adv = X_adv + alpha * np.sign(input_gradient(model, X_adv, labels))
        X_adv = np.clip(np.clip(X_adv, X - eps, X + eps), 0, 1)
    return X_adv


def random_sign(X, eps: float, seed: int = 0):
    """x + eps * (random signs), clipped to [0, 1]: a perturbation of the same size as FGSM but in a random direction."""
    rng = np.random.default_rng(seed)
    return np.clip(X + eps * rng.choice([-1.0, 1.0], size=X.shape), 0, 1)


def relative_change(model, X, X_perturbed):
    """||h_l(x + r) - h_l(x)|| / ||h_l(x)|| for every layer l of the model (model.layer_names), averaged over the images."""
    h, h_perturbed = model.forward(X), model.forward(X_perturbed)
    return np.array([np.mean(np.linalg.norm(b - a, axis=1) / np.maximum(np.linalg.norm(a, axis=1), 1e-12)) for a, b in zip(h, h_perturbed)])


def attack_table(model, X, labels, eps_values, n_pgd_steps: int = 5, seed: int = 0):
    """Test accuracy under FGSM, PGD (n_pgd_steps steps of alpha = eps / 2) and random signs, for every eps."""
    table = {"FGSM": [], "PGD": [], "random signs": []}
    for eps in eps_values:
        table["FGSM"].append(accuracy(model, fgsm(model, X, labels, eps), labels))
        table["PGD"].append(accuracy(model, pgd(model, X, labels, eps, alpha=eps / 2, n_steps=n_pgd_steps), labels))
        table["random signs"].append(accuracy(model, random_sign(X, eps, seed=seed), labels))
    return table


def print_attack_table(table, clean, eps_values, name: str):
    print(f"{name}: clean test accuracy {clean:.1%}")
    print("    eps:        " + "   ".join(f"{eps:6.2f}" for eps in eps_values))
    for attack, accs in table.items():
        print(f"    {attack:13s}" + "   ".join(f"{a:6.1%}" for a in accs))


def linear_argument(n_values, eps: float = 1.0, n_draws: int = 2000, seed: int = 0):
    """For i.i.d. Gaussian weights w with n components: the worst change eps ||w||_1, the standard deviation of w.eta over random
    signs eta_i = +-eps (the formula eps ||w||_2 and the value measured over n_draws draws) and the ratio ||w||_1 / ||w||_2."""
    rng = np.random.default_rng(seed)
    rows = []
    for n in n_values:
        w = rng.normal(size=n)
        eta = eps * rng.choice([-1.0, 1.0], size=(n_draws, n))
        rows.append((n, eps * np.abs(w).sum(), eps * np.linalg.norm(w), float(np.std(eta @ w)), np.abs(w).sum() / np.linalg.norm(w)))
    return rows


n_values = [16, 64, 256, 1024, 4096, 16384]
rows = linear_argument(n_values, eps=1.0)
print("     n   eps||w||_1   eps||w||_2   measured std   ratio ||w||_1/||w||_2   sqrt(n)")
for n, worst, l2, std_measured, ratio in rows:
    print(f"{n:6d}   {worst:10.2f}   {l2:10.2f}   {std_measured:12.2f}   {ratio:21.2f}   {np.sqrt(n):7.2f}")

plt.figure(figsize=(6.5, 4.2))
plt.loglog(n_values, [row[4] for row in rows], "o-", color=TEAL, label="measured ratio, i.i.d. Gaussian weights")
plt.loglog(n_values, np.sqrt(n_values), "--", color=GREY, label=r"the bound $\sqrt{n}$")
plt.xlabel("number of inputs $n$")
plt.ylabel(r"$\varepsilon\|w\|_1 \,/\, \varepsilon\|w\|_2$")
plt.title("gradient-sign change against the standard deviation of random signs")
plt.legend(fontsize=8)
plt.show()

In [ ]:
eps_values = [0.02, 0.05, 0.1]
clean_mlp = accuracy(mlp, X_test, y_test)
table_mlp = attack_table(mlp, X_test, y_test, eps_values)
print_attack_table(table_mlp, clean_mlp, eps_values, "the MLP, standard training")

attack_colours = {"FGSM": ORANGE, "PGD": "#7f2704", "random signs": GREY}
plt.figure(figsize=(6.5, 4))
for attack, accs in table_mlp.items():
    plt.plot([0] + eps_values, [clean_mlp] + accs, "o-", color=attack_colours[attack], label=attack)
plt.xlabel(r"perturbation bound $\varepsilon$ (pixels in [0, 1])")
plt.ylabel("test accuracy")
plt.ylim(-0.02, 1.02)
plt.title("accuracy of the MLP under the three perturbations")
plt.legend()
plt.show()

In [ ]:
# one example: the image x, the FGSM perturbation r and x + r, with the predicted labels and probabilities
eps_show = 0.05
X_adv = fgsm(mlp, X_test, y_test, eps_show)
pred_clean, pred_adv = predict(mlp, X_test), predict(mlp, X_adv)
candidates = np.flatnonzero((pred_clean == y_test) & (pred_adv != y_test))    # correct before, wrong after
P_true = mlp.forward(X_test)[-1][np.arange(len(y_test)), y_test]
k = int(candidates[np.argmin(np.abs(P_true[candidates] - 0.9))]) if len(candidates) else 0   # one classified with probability about 0.9
x, r = X_test[k], X_adv[k] - X_test[k]
P_clean, P_adv = mlp.forward(x[None])[-1][0], mlp.forward(X_adv[k][None])[-1][0]
scale = int(round(0.25 / eps_show))                                           # r is shown x scale around mid-grey

fig, axes = plt.subplots(1, 3, figsize=(9.5, 3.6))
panels = [(x, f"$x$: {CLASS_NAMES[pred_clean[k]]} ({P_clean[pred_clean[k]]:.2f})", TEAL if pred_clean[k] == y_test[k] else ORANGE),
          (0.5 + scale * r, rf"$r$, shown $\times${scale} around mid-grey", "black"),
          (x + r, f"$x + r$: {CLASS_NAMES[pred_adv[k]]} ({P_adv[pred_adv[k]]:.2f})", TEAL if pred_adv[k] == y_test[k] else ORANGE)]
for ax, (image, title, colour) in zip(axes, panels):
    ax.imshow(image.reshape(SIZE, SIZE), cmap="gray", vmin=0, vmax=1)
    ax.set_title(title, color=colour, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(rf"test image {k}, true label: {CLASS_NAMES[y_test[k]]}; FGSM on the MLP with $\varepsilon$ = {eps_show}")
plt.show()

# the linear argument on this image: the gradient g, the predicted and the actual change of the loss
g = input_gradient(mlp, x[None], y_test[k:k + 1])[0]
l1, l2 = np.abs(g).sum(), np.linalg.norm(g)
loss_x = losses_of(P_clean[None], y_test[k:k + 1])[0]
loss_adv = losses_of(P_adv[None], y_test[k:k + 1])[0]
x_rnd = random_sign(x[None], eps_show, seed=0)
P_rnd = mlp.forward(x_rnd)[-1][0]
loss_rnd = losses_of(P_rnd[None], y_test[k:k + 1])[0]
print(f"loss at x: {loss_x:.3f};  ||g||_1 = {l1:.3g}, ||g||_2 = {l2:.3g}, ratio {l1 / l2:.1f} (the bound sqrt(n) = {np.sqrt(len(x)):.1f})")
print(f"FGSM:         linear prediction of the increase eps ||g||_1 = {eps_show * l1:.3f}; actual loss at x + r {loss_adv:.3f} (increase {loss_adv - loss_x:+.3f})")
print(f"random signs: standard deviation of the linear term eps ||g||_2 = {eps_show * l2:.3g}; actual loss {loss_rnd:.3f} (change {loss_rnd - loss_x:+.3f}); "
      f"predicted label {CLASS_NAMES[int(P_rnd.argmax())]}")

In [ ]:
# how the perturbation changes through the layers of the MLP (the notes' Figure 19, p. 70)
layer_names = mlp.layer_names
X_sub, y_sub = X_test[:300], y_test[:300]
changes = {"adversarial (PGD, 5 steps)": relative_change(mlp, X_sub, pgd(mlp, X_sub, y_sub, eps_show, alpha=eps_show / 2)),
           "adversarial (FGSM)": relative_change(mlp, X_sub, fgsm(mlp, X_sub, y_sub, eps_show)),
           "random signs": relative_change(mlp, X_sub, random_sign(X_sub, eps_show))}
print(f"relative change per layer of the MLP, eps = {eps_show}, mean over {len(X_sub)} test images")
widths = [max(12, len(name) + 3) for name in layer_names]
print(f"{'':28s}" + "".join(f"{name:>{wd}s}" for name, wd in zip(layer_names, widths)))
for name, values in changes.items():
    print(f"{name:28s}" + "".join(f"{v:{wd}.3f}" for v, wd in zip(values, widths)))

plt.figure(figsize=(6.5, 4))
styles = {"adversarial (PGD, 5 steps)": (ORANGE, "-"), "adversarial (FGSM)": (ORANGE, "--"), "random signs": (GREY, "-")}
for name, values in changes.items():
    plt.plot(layer_names, values, "o" + styles[name][1], color=styles[name][0], label=name)
plt.xlabel("layer $l$")
plt.ylabel(r"relative change $\|h_l(x + r) - h_l(x)\| / \|h_l(x)\|$")
plt.title(rf"the perturbation through the layers of the MLP ($\varepsilon$ = {eps_show})")
plt.legend()
plt.show()

In [ ]:
# adversarial training of a fresh MLP: FGSM examples at eps = 0.05 are added to every minibatch
mlp_adv = LayerNetwork([SIZE * SIZE, 64, 32, 3], seed=0)
history_adv = train_model(mlp_adv, X_train, y_train, n_passes=20, batch_size=50, rate=0.05, momentum=0.9, seed=0, adversarial_eps=0.05)
clean_adv = accuracy(mlp_adv, X_test, y_test)
table_adv = attack_table(mlp_adv, X_test, y_test, eps_values)
print_attack_table(table_mlp, clean_mlp, eps_values, "before (standard training)")
print_attack_table(table_adv, clean_adv, eps_values, "after (adversarial training with FGSM examples at eps = 0.05)")

plt.figure(figsize=(6.5, 4))
plt.plot([0] + eps_values, [clean_mlp] + table_mlp["FGSM"], "o-", color=ORANGE, label="standard training")
plt.plot([0] + eps_values, [clean_adv] + table_adv["FGSM"], "o-", color=TEAL, label="adversarial training")
plt.xlabel(r"perturbation bound $\varepsilon$ of the FGSM attack")
plt.ylabel("test accuracy under FGSM")
plt.ylim(-0.02, 1.02)
plt.title("accuracy under FGSM before and after adversarial training")
plt.legend()
plt.show()

### Question 10.5 (4 points)

- Report the ratio $\|w\|_1 / \|w\|_2$ for i.i.d. Gaussian weights at $n = 16$, $256$ and $4096$ next to the bound $\sqrt n$. Use the linear argument to explain why random signs of the same size $\varepsilon$ should change the loss, and so the accuracy, much less than the gradient signs. **(1 point)**
- Report the test accuracy of the MLP under FGSM, 5-step PGD ($\alpha = \varepsilon/2$) and random signs at $\varepsilon \in \{0.02, 0.05, 0.1\}$, and show one example triple $(x, r, x + r)$ with the predicted labels. **(1 point)**
- Plot the relative change of every layer for the adversarial and for the random perturbation, and explain the difference. **(1 point)**
- Retrain with adversarial examples (`adversarial_eps=0.05`) and report the clean and the FGSM test accuracy before and after. Which step of min-max training does the FGSM example implement? **(1 point)**

In [ ]:
# Your code for Question 10.5

**Your answers for Question 10.5:**